# 🛠️ Practice Project Template: End-to-End House Price Regression Pipeline
## Regression Modeling, Residual Diagnostics & Regularization

Predicting real estate prices is the classic regression benchmark. It tests your ability to handle non-linearities, scale features, tune regularized linear models (Ridge/Lasso), and benchmark against Gradient Boosted Decision Trees.

---

### 📋 Project Roadmap:
1. **Data Exploration & Target Distribution**: Check for skewness in the target variable `price`.
2. **Feature Transformations**: Scaling numerical features and encoding location categories.
3. **Model Exploration**:
   - Ordinary Least Squares (OLS)
   - Ridge Regression ($L_2$ Regularization)
   - Lasso Regression ($L_1$ Feature Selection)
   - Gradient Boosting Regressor
4. **Model Evaluation**: $R^2$ Score, Mean Absolute Error (MAE), Root Mean Squared Error (RMSE).
5. **Residual Diagnostics**: Residual vs Fitted plot and checking for heteroskedasticity!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("house_pricing_data.csv")
print(f"Housing dataset: {df.shape}")
df.head()


---
## Step 1: Train/Test Split & Feature Setup


In [ ]:
X = df.drop('price', axis=1)
y = df['price']

num_features = ['sqft', 'bedrooms', 'bathrooms', 'age']
cat_features = ['neighborhood']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} homes, Test: {X_test.shape[0]} homes")


---
## Step 2: [YOUR TURN] Preprocessing Pipeline
**Instructions:**
- Create `StandardScaler` for `num_features`.
- Create `OneHotEncoder` for `cat_features`.
- Combine inside `ColumnTransformer`.


In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
])
print("Preprocessor ready!")


---
## Step 3: [YOUR TURN] Model Comparison
**Instructions:**
- Train and evaluate: Linear Regression, Ridge ($\alpha=10$), Lasso ($\alpha=100$), and Gradient Boosting Regressor.
- Compare RMSE, MAE, and $R^2$ on the test set.


In [ ]:
regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge (L2)": Ridge(alpha=10.0),
    "Lasso (L1)": Lasso(alpha=100.0),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=42)
}

results = []
for name, model in regressors.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "RMSE ($)": round(rmse, 2),
        "MAE ($)": round(mae, 2),
        "R2 Score": round(r2, 4)
    })

print("=== REGRESSION PERFORMANCE COMPARISON ===")
print(pd.DataFrame(results).to_string(index=False))


---
## Step 4: Residual Diagnostics
A healthy regression model should have residuals normally distributed around 0 with constant variance (homoskedasticity).


In [ ]:
gbr_pipe = Pipeline([('prep', preprocessor), ('reg', GradientBoostingRegressor(n_estimators=100, random_state=42))])
gbr_pipe.fit(X_train, y_train)
preds = gbr_pipe.predict(X_test)
residuals = y_test - preds

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(preds, residuals, alpha=0.6, color='steelblue')
plt.axhline(0, color='crimson', linestyle='--')
plt.title("Residuals vs Fitted Values")
plt.xlabel("Predicted Price ($)")
plt.ylabel("Residual Error ($)")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(residuals, bins=25, color='mediumpurple', edgecolor='black', alpha=0.7)
plt.title("Residual Error Distribution")
plt.xlabel("Residual ($)")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 🏆 Practice Challenge Extensions:
1. **Target Log Transform**: Try training on $\log(\text{price})$ using `TransformedTargetRegressor` and evaluate whether RMSE improves.
2. **Polynomial Features**: Add interaction terms between `sqft` and `neighborhood`!
